# Word2Vec → Item2Vec → простий рекомендаційний рушій

У цьому notebook ми пройдемо шлях від **ідеї Word2Vec** до дуже простого рекомендаційного рушія на **MovieLens**.

Головна мета — зрозуміти не лише код, а **чому це працює**.

План:

1. Що таке embedding.
2. Як Word2Vec навчається на контексті.
3. Різниця між **CBOW** і **Skip-Gram**.
4. Як перенести цю ідею зі слів на товари / фільми.
5. Побудувати **Item2Vec** на MovieLens через `gensim`.
6. Знайти схожі фільми.
7. Побудувати просту рекомендацію для користувача.
8. Виконати невелике самостійне завдання.

# 1. Що таке embedding?

Комп'ютер не розуміє слово `"кіт"` або назву фільму `"Toy Story"` так, як людина.

Йому потрібні числа.

Найпростіший варіант — дати кожному об'єкту окремий ID:

```text
кіт      -> 0
собака   -> 1
молоко   -> 2
вода     -> 3
```

Або one-hot вектор:

```text
кіт      = [1, 0, 0, 0]
собака   = [0, 1, 0, 0]
молоко   = [0, 0, 1, 0]
вода     = [0, 0, 0, 1]
```

Проблема: такі вектори лише кажуть, що об'єкти **різні**.

Вони не кажуть, що:

```text
кіт ≈ собака
```

і що:

```text
кіт
```

має значно менше спільного зі словом:

```text
бетон
```

Embedding — це маленький щільний вектор, який модель **вивчає сама**:

```text
кіт      = [ 0.72,  0.81, -0.24]
собака   = [ 0.68,  0.77, -0.19]

молоко   = [-0.64,  0.13,  0.82]
вода     = [-0.59,  0.16,  0.75]
```

Якщо два вектори близькі — об'єкти в певному сенсі теж схожі.

# 2. Головна ідея Word2Vec

Word2Vec не читає словникові визначення.

Він дивиться на **контекст**.

Наприклад:

```text
кіт їсть корм
собака їсть корм

кіт п'є воду
собака п'є воду
```

Слова `кіт` і `собака` часто з'являються біля однакових слів.

Тому модель поступово навчає для них схожі вектори.

Головна ідея:

> **Об'єкти, які зустрічаються в схожих контекстах, отримують схожі embeddings.**

# 3. Що таке context window?

Візьмемо речення:

```text
я дуже люблю пити гарячу каву вранці
```

Нехай центральне слово:

```text
пити
```

і `window = 2`.

Тоді контекстом будуть два слова ліворуч і два слова праворуч:

```text
я   дуже   люблю   [ПИТИ]   гарячу   каву   вранці
     ↑       ↑                  ↑       ↑
     └──────────── CONTEXT ─────────────┘
```

Контекст:

```text
дуже
люблю
гарячу
каву
```

Це вікно рухається по всьому тексту.

# 4. CBOW

**CBOW = Continuous Bag of Words**

CBOW отримує **слова навколо** і намагається передбачити **центральне слово**.

```text
дуже + люблю + гарячу + каву
               ↓
              пити
```

Тобто:

```text
CONTEXT → TARGET
```

Схематично:

```text
дуже ───┐
люблю ──┤
гарячу ─┤ → embeddings → average → prediction → "пити"
каву ───┘
```

Модель бере embedding кожного контекстного слова, об'єднує їх і намагається правильно вгадати слово посередині.

### Чому `Bag of Words`?

У класичному CBOW порядок контекстних слів майже не важливий.

Наприклад, модель може усереднити:

```text
embedding("дуже")
embedding("люблю")
embedding("гарячу")
embedding("каву")
```

Тому контекст сприймається приблизно як "мішок слів".

# 5. Skip-Gram

**Skip-Gram робить протилежне.**

Він отримує центральне слово і намагається передбачити слова навколо.

```text
              пити
          ↙    ↓    ↘
      люблю  гарячу  каву
```

Тобто:

```text
TARGET → CONTEXT
```

На практиці один контекст перетворюється на багато тренувальних пар.

Наприклад:

```text
я дуже люблю пити гарячу каву
```

для слова `пити` та `window=2`:

```text
(пити, дуже)
(пити, люблю)
(пити, гарячу)
(пити, каву)
```

Саме такі пари бачить модель під час навчання.

# 6. Що насправді навчається?

На початку вектори випадкові.

Умовно:

```text
кіт = [-0.12, 0.43, 0.07]
```

Модель робить прогноз, помиляється і змінює числа.

```text
випадковий embedding
        ↓
prediction
        ↓
error / loss
        ↓
оновлення ваг
        ↓
кращий embedding
```

Після великої кількості прикладів слова зі схожими контекстами стають близькими у векторному просторі.

Наприклад:

```text
кіт      -> [0.71, 0.83, -0.21]
собака   -> [0.68, 0.79, -0.18]
```

# 7. Навіщо нам це для рекомендацій?

Тепер найважливіший перехід.

Word2Vec працює приблизно так:

```text
слово → слова, які часто знаходяться поруч
```

Але слово не обов'язково має бути справжнім словом.

Замінимо:

```text
слово      → movieId
речення    → історія фільмів одного користувача
контекст   → інші фільми цього користувача
```

Наприклад, користувач дивився або високо оцінив:

```text
Toy Story
Finding Nemo
Monsters, Inc.
Shrek
Ice Age
```

Для Item2Vec це можна представити як "речення":

```python
["1", "6377", "3114", "4306", "5218"]
```

Тоді Skip-Gram навчається на парах на кшталт:

```text
Toy Story     → Finding Nemo
Toy Story     → Monsters, Inc.
Finding Nemo  → Shrek
...
```

Якщо два фільми часто зустрічаються у профілях однакових користувачів, їх embeddings стають близькими.

Це і є ідея **Item2Vec**.

# 8. Важлива різниця

Ми **не навчаємо модель на назвах фільмів**.

Ось це було б неправильно для нашої задачі:

```python
["Toy", "Story", "1995"]
```

Нам потрібна поведінка користувачів:

```text
користувач 1:
[Movie A, Movie B, Movie C]

користувач 2:
[Movie A, Movie C, Movie D]

користувач 3:
[Movie B, Movie C, Movie E]
```

Тоді модель вивчає:

> які фільми люди часто споживають разом?

# 9. Практична частина: MovieLens + gensim

Використаємо реальний датасет **MovieLens latest-small**.

Він містить приблизно 100 тисяч оцінок фільмів від сотень користувачів.

Будемо використовувати:

- `pandas` — робота з таблицями;
- `gensim` — готова реалізація Word2Vec;
- `matplotlib` — графіки;
- `scikit-learn` — PCA для 2D-візуалізації.

> Якщо бібліотеки вже встановлені, цей блок можна пропустити.

In [ ]:
%pip install -q pandas gensim scikit-learn matplotlib

# 10. Завантажуємо MovieLens

MovieLens розповсюджується GroupLens.

Ми завантажимо маленьку версію датасету прямо з офіційного сайту.

In [ ]:
import io
import zipfile
import urllib.request

import pandas as pd

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

# Завантажуємо zip-файл у пам'ять
response = urllib.request.urlopen(url)
data = response.read()

# Відкриваємо архів
archive = zipfile.ZipFile(io.BytesIO(data))

# Читаємо дві потрібні таблиці
ratings = pd.read_csv(archive.open("ml-latest-small/ratings.csv"))
movies = pd.read_csv(archive.open("ml-latest-small/movies.csv"))

print("ratings:", ratings.shape)
print("movies:", movies.shape)

ratings.head()

У таблиці `ratings`:

- `userId` — користувач;
- `movieId` — фільм;
- `rating` — оцінка від 0.5 до 5;
- `timestamp` — час оцінки.

Подивимося на назви фільмів.

In [ ]:
movies.head()

# 11. Залишимо лише позитивні взаємодії

Item2Vec зазвичай будується на поведінці:

```text
користувач взаємодіяв з item
```

У MovieLens є оцінки.

Щоб логіка була простою, вважатимемо позитивною взаємодією:

```text
rating >= 4.0
```

Тобто користувач фільм скоріше **вподобав**.

In [ ]:
positive_ratings = ratings[ratings["rating"] >= 4.0].copy()

print("Усі оцінки:", len(ratings))
print("Позитивні оцінки:", len(positive_ratings))

positive_ratings.head()

# 12. Створюємо "речення" для Item2Vec

Для Word2Vec речення може виглядати так:

```python
["я", "люблю", "дивитися", "кіно"]
```

Для Item2Vec нашим "реченням" буде список фільмів одного користувача:

```python
["1", "260", "1196", "1210", ...]
```

Чому перетворюємо `movieId` у рядки?

`gensim.Word2Vec` очікує послідовності токенів. Рядкові ID зручно використовувати як токени.

In [ ]:
# Групуємо позитивно оцінені фільми за користувачем
user_movies = (
    positive_ratings
    .groupby("userId")["movieId"]
    .apply(lambda x: [str(movie_id) for movie_id in x])
    .tolist()
)

print("Кількість користувацьких послідовностей:", len(user_movies))

# Покажемо перші 3
for sequence in user_movies[:3]:
    print(sequence[:15])

# 13. Навчаємо Item2Vec

Використаємо `gensim.models.Word2Vec`.

Найважливіші параметри:

```python
vector_size=50
```

Кожен фільм отримає вектор з 50 чисел.

```python
window=5
```

Модель дивиться на сусідні items у межах вікна.

```python
sg=1
```

`sg=1` означає **Skip-Gram**.

`sg=0` означав би **CBOW**.

```python
min_count=2
```

Ігноруємо фільми, які зустрілися лише один раз.

```python
epochs=30
```

Кілька разів проходимо по наших послідовностях.

> Для Item2Vec часто використовують саме Skip-Gram, бо нас цікавить зв'язок `item → інші items`.

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=user_movies,
    vector_size=50,
    window=5,
    min_count=2,
    sg=1,           # 1 = Skip-Gram
    workers=4,
    epochs=30,
    seed=42
)

print("Кількість фільмів у словнику моделі:", len(model.wv))

# 14. Що ми отримали?

Тепер кожен `movieId` має embedding.

Наприклад:

In [ ]:
movie_id = "1"  # Toy Story

vector = model.wv[movie_id]

print("Розмір embedding:", len(vector))
print(vector[:10])

Ми отримали приблизно:

```text
Movie ID
   ↓
50 чисел
   ↓
embedding
```

Тепер можемо порівнювати фільми через cosine similarity.

# 15. Допоміжні словники ID ↔ назва

Щоб не дивитися на `movieId`, створимо словники:

```text
movieId → title

title → movieId
```

In [ ]:
id_to_title = dict(zip(movies["movieId"].astype(str), movies["title"]))
title_to_id = dict(zip(movies["title"], movies["movieId"].astype(str)))

print(id_to_title["1"])

# 16. Знайдемо схожі фільми

`gensim` має готовий метод:

```python
model.wv.most_similar(...)
```

Він знаходить найближчі embeddings.

In [ ]:
movie_id = "1"

print("Фільм:", id_to_title[movie_id])
print()

similar_movies = model.wv.most_similar(movie_id, topn=10)

for similar_id, similarity in similar_movies:
    print(f"{id_to_title.get(similar_id, similar_id):50s} similarity={similarity:.3f}")

## Що означає `similarity`?

Це cosine similarity між двома embeddings.

Спрощено:

```text
1.0   → дуже схожі напрямки
0.0   → майже не пов'язані
-1.0  → протилежні напрямки
```

У рекомендаційних embeddings зазвичай нас цікавлять найбільші значення.

# 17. Зробимо зручну функцію

Тепер можна передати назву фільму й отримати схожі.

In [ ]:
def recommend_similar_movies(title, topn=10):
    # Перевіряємо, чи є така точна назва у таблиці
    if title not in title_to_id:
        print("Не знайшов таку точну назву фільму.")
        return

    movie_id = title_to_id[title]

    # Перевіряємо, чи цей фільм потрапив у словник Word2Vec
    if movie_id not in model.wv:
        print("Цей фільм має замало позитивних взаємодій для нашої моделі.")
        return

    print("Схожі на:", title)
    print()

    for similar_id, similarity in model.wv.most_similar(movie_id, topn=topn):
        print(f"{id_to_title.get(similar_id, similar_id):50s} {similarity:.3f}")

In [ ]:
recommend_similar_movies("Toy Story (1995)")

# 18. Як отримати рекомендацію не від одного фільму, а від декількох?

Уявімо, що користувач любить:

```text
Toy Story
Finding Nemo
Shrek
```

У кожного є embedding:

```text
Toy Story     → v1
Finding Nemo  → v2
Shrek         → v3
```

Можемо усереднити ці вектори:

\[
user\_vector = \frac{v_1 + v_2 + v_3}{3}
\]

Так отримаємо дуже простий **embedding профілю користувача**.

Потім знайдемо фільми, embeddings яких найближчі до цього вектора.

In [ ]:
import numpy as np

liked_titles = [
    "Toy Story (1995)",
    "Finding Nemo (2003)",
    "Shrek (2001)"
]

liked_ids = []

for title in liked_titles:
    movie_id = title_to_id.get(title)

    if movie_id is not None and movie_id in model.wv:
        liked_ids.append(movie_id)

print("Фільми користувача:")
for movie_id in liked_ids:
    print("-", id_to_title[movie_id])

In [ ]:
# Беремо embeddings усіх вподобаних фільмів
vectors = [model.wv[movie_id] for movie_id in liked_ids]

# Усереднюємо
user_vector = np.mean(vectors, axis=0)

print("Розмір user embedding:", user_vector.shape)
print(user_vector[:10])

Тепер шукаємо найближчі фільми до `user_vector`.

Важливо: треба прибрати фільми, які користувач уже вибрав.

In [ ]:
recommendations = model.wv.similar_by_vector(user_vector, topn=20)

print("Рекомендації:")
print()

shown = 0

for movie_id, similarity in recommendations:
    # Не рекомендуємо те, що вже було у профілі
    if movie_id in liked_ids:
        continue

    print(f"{id_to_title.get(movie_id, movie_id):50s} {similarity:.3f}")
    shown += 1

    if shown == 10:
        break

# 19. Візуалізація embeddings

Наш embedding має 50 вимірів.

Людина не може намалювати 50-вимірний простір.

Тому зменшимо його до 2 вимірів за допомогою PCA.

Візьмемо один фільм та найближчі до нього фільми.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

base_movie = "Toy Story (1995)"
base_id = title_to_id[base_movie]

neighbors = model.wv.most_similar(base_id, topn=12)

ids_to_plot = [base_id] + [movie_id for movie_id, _ in neighbors]
titles_to_plot = [id_to_title[movie_id] for movie_id in ids_to_plot]

vectors_to_plot = np.array([model.wv[movie_id] for movie_id in ids_to_plot])

# 50D -> 2D
pca = PCA(n_components=2)
points = pca.fit_transform(vectors_to_plot)

plt.figure(figsize=(12, 8))
plt.scatter(points[:, 0], points[:, 1])

for i, title in enumerate(titles_to_plot):
    plt.annotate(
        title,
        (points[i, 0], points[i, 1]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title("Item2Vec embeddings: фільми біля Toy Story")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.grid(alpha=0.2)
plt.show()

# 20. Що ми щойно зробили?

Повний pipeline:

```text
MovieLens ratings
        ↓
залишили позитивні взаємодії
        ↓
для кожного user отримали список movieId
        ↓
списки стали "реченнями"
        ↓
Word2Vec Skip-Gram
        ↓
кожен movieId отримав embedding
        ↓
cosine similarity
        ↓
схожі фільми / рекомендації
```

Тобто Word2Vec перетворився на Item2Vec лише завдяки зміні значення "токена":

```text
Word2Vec:
token = слово

Item2Vec:
token = item
```

# 21. Важливі обмеження нашого прикладу

Це навчальний recommendation engine, а не production-система.

### 1. Ми втратили оцінку

Ми перетворили:

```text
rating >= 4
```

на простий факт позитивної взаємодії.

Тобто `5.0` і `4.0` для моделі однакові.

### 2. Порядок тут не зовсім природний

У тексті порядок слів має сенс.

Для фільмів можна використовувати порядок взаємодій за часом, але в нашому простому прикладі головна ідея — **co-occurrence** фільмів у профілях користувачів.

### 3. Cold Start

Якщо новий фільм ще ніхто не оцінював, Item2Vec нічого про нього не знає.

### 4. Популярність

Популярні items отримують набагато більше сигналу.

### 5. Це item-to-item модель

Модель чудово вчить схожість між items, але сама по собі не є повною сучасною recommender system.

# 22. CBOW чи Skip-Gram для Item2Vec?

Можна спробувати обидва.

У `gensim`:

```python
sg=0
```

означає CBOW:

```text
сусідні items → центральний item
```

а:

```python
sg=1
```

означає Skip-Gram:

```text
центральний item → сусідні items
```

Для Item2Vec дуже природно використовувати Skip-Gram:

```text
Movie A → Movie B
Movie A → Movie C
Movie A → Movie D
```

Але це не означає, що CBOW ніколи не можна використовувати.

Найкраще — **перевіряти на даних**.

# Самостійне завдання

Тепер потрібно **не писати Word2Vec самостійно**, а правильно використати готовий `gensim`.

Мета завдання:

> Побудувати простий Item2Vec recommendation engine на MovieLens та пояснити отриманий результат.

Використовуйте код вище як основу.

## Завдання 1. Підготуйте дані

Залиште лише оцінки:

```python
rating >= 4.0
```

Потім для кожного `userId` сформуйте список `movieId`.

У результаті повинна вийти структура приблизно такого типу:

```python
[
    ["1", "260", "1196", "1210"],
    ["296", "318", "356"],
    ["50", "527", "593", "608"],
    ...
]
```

### Перевірте себе

Виведіть:

1. кількість користувачів;
2. перші 3 послідовності;
3. довжину однієї з послідовностей.

In [ ]:
# TODO: Завдання 1

# 1. Відібрати rating >= 4.0
# positive_ratings = ...

# 2. Згрупувати movieId за userId
# user_movies = ...

# 3. Вивести кількість користувачів та кілька прикладів

## Завдання 2. Навчіть Item2Vec

Створіть модель:

```python
Word2Vec(
    sentences=user_movies,
    vector_size=50,
    window=5,
    min_count=2,
    sg=1,
    epochs=30
)
```

Після навчання:

1. виведіть кількість фільмів у словнику;
2. виберіть будь-який `movieId`;
3. виведіть перші 10 чисел його embedding.

### Питання

Що означає:

```python
vector_size=50
```

своїми словами?

In [ ]:
# TODO: Завдання 2

# from gensim.models import Word2Vec

# model = Word2Vec(...)

# print(...)

## Завдання 3. Знайдіть схожі фільми

Виберіть **один відомий фільм**, який вам знайомий.

Наприклад:

```text
Toy Story (1995)
Matrix, The (1999)
Shrek (2001)
Finding Nemo (2003)
```

Знайдіть для нього **10 найближчих фільмів** через:

```python
model.wv.most_similar(...)
```

Виведіть:

```text
назва фільму | similarity
```

### Дайте коротку відповідь

Після списку напишіть 2–4 речення:

- чи здаються рекомендації логічними;
- які фільми вас здивували;
- чому, на вашу думку, модель поставила їх поруч.

In [ ]:
# TODO: Завдання 3

# title = ...

# movie_id = ...
# similar_movies = ...

# for ...

## Завдання 4. Побудуйте просту рекомендацію для користувача

Виберіть **3 фільми**, які подобаються умовному користувачу.

Наприклад:

```python
liked_titles = [
    "Toy Story (1995)",
    "Finding Nemo (2003)",
    "Shrek (2001)"
]
```

Зробіть:

1. знайдіть embeddings цих трьох фільмів;
2. порахуйте їх середній вектор через `np.mean`;
3. знайдіть найближчі фільми до цього вектора;
4. приберіть ті фільми, які вже були у `liked_titles`;
5. покажіть **5 рекомендацій**.

In [ ]:
# TODO: Завдання 4

# liked_titles = [
#     "...",
#     "...",
#     "..."
# ]

# liked_ids = ...
# vectors = ...
# user_vector = ...

# recommendations = ...

## Завдання 5. Простий експеримент

Навчіть **другу модель**, але змініть:

```python
window=2
```

замість:

```python
window=5
```

Для одного й того самого фільму знайдіть `top-5` схожих фільмів у двох моделях.

Порівняйте:

```text
window = 2
```

та

```text
window = 5
```

### Дайте коротку відповідь

Чи змінилися рекомендації?

Спробуйте пояснити чому.

Підказка:

> `window` визначає, наскільки широкий контекст бачить Word2Vec.

In [ ]:
# TODO: Завдання 5

# model_window_2 = Word2Vec(
#     sentences=user_movies,
#     vector_size=50,
#     window=2,
#     min_count=2,
#     sg=1,
#     epochs=30,
#     seed=42
# )

# Порівняйте top-5 з основною моделлю

# Що потрібно здати

У результаті має бути один notebook, у якому видно:

- підготовку MovieLens;
- список movieId для кожного користувача;
- навчання Item2Vec;
- embedding хоча б одного фільму;
- `top-10` схожих фільмів;
- рекомендації для профілю з 3 фільмів;
- порівняння `window=2` та `window=5`;
- короткі текстові висновки.

## Головне

Не потрібно писати складний recommendation engine.

Потрібно зрозуміти і показати ланцюжок:

```text
поведінка користувачів
        ↓
послідовності items
        ↓
Word2Vec / Skip-Gram
        ↓
item embeddings
        ↓
cosine similarity
        ↓
recommendations
```

Якщо цей ланцюжок зрозумілий — основна мета роботи виконана.